# Overview — start here

**Question.** How sensitive was eight years of ZTF (2018 March – 2026 March) to optical kilonovae, and what rate
limit follows if no candidate passes the search criteria?

Every number on this page is computed from the results snapshot. Each one points to the notebook and paper section
where it is examined. Decisions and their reasons (models dropped, suites retired, choices of window and
criteria) are in [`docs/decisions.md`](../docs/decisions.md).

| Notebook | Question it answers | Paper |
|---|---|---|
| `01_limits` | What are ⟨VT⟩ and the rate limits, and how far does ZTF see? | §4, Fig. 3, Fig. 4, Table 1 |
| `02_survey_time_and_previous_searches` | How did sensitivity build up, and how do we compare with earlier ZTF limits? | §4.1, Fig. 1 |
| `03_viewing_angle` | How much does the orientation matter? | §5, Fig. 2 |
| `04_systematics` | How robust are the limits (MC noise, z_max, parameters, light curves, criteria, population)? | §6, §8, Fig. 5 |
| `05_criteria_companion` | How does each selection criterion change ⟨VT⟩? | companion paper, Fig. 6 |
| `paper_figures` | Writes `paper/figures/` and Table 1 | — |
| `telemetry_job_resources` | Cluster usage of the runs | — |

The notebooks from before this restructure (one per simulation suite) are in `defunct/notebooks_by_suite/`.
They still run, and they hold the longer explanations of each figure.

In [1]:
# ---------------------------------------------------------------- settings
SNAPSHOT = "latest"      # "latest" | time stamp or prefix | "newest" | path/to/snapshot.zip | "live"
SAVE_FIGURES = False     # write PDF+PNG of each figure to figures/00/
# ---------------------------------------------------------------------------
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "knrates").is_dir())
sys.path.insert(0, str(ROOT))
for _m in [m for m in list(sys.modules) if m.split(".")[0] == "knrates"]:
    del sys.modules[_m]   # always import the repository copy (a kernel may predate a git pull)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from knrates import notebook as NB, figures as F, analysis as A, plotting as P, literature as L
from knrates.stats import summarize_runs

ctx = NB.setup(SNAPSHOT, save_figures=SAVE_FIGURES, fig_dir="figures/00", quiet=True)
runs, lim, M = ctx.runs, ctx.lim, ctx.models
FIX, ISO = "Bu2026Fixed", "Bu2026Vary"

## What is in the snapshot

In [2]:
from knrates import load_snapshot
raw = load_snapshot(SNAPSHOT)      # unfiltered: every suite and model
inv = []
for t, label in [("suite01_runs", "01 fixed-parameter ensemble"), ("suite07_runs", "07 GW170817 parameter set"),
                 ("suite03_runs", "03 regimes x populations"), ("suite02_candidate_runs", "02 criteria-sweep candidate sets")]:
    d = raw.get(t)
    if d is None or d.empty:
        continue
    key = "population_type" if "population_type" in d else "model"
    for m, g in d.groupby(key):
        inv.append(dict(suite=label, model=m, realisations=len(g),
                        injected=g["n_simulated"].sum() if "n_simulated" in g else g.get("n_transients", pd.Series([np.nan])).sum()))
inv = pd.DataFrame(inv)
inv["used in main analysis"] = inv.model.isin(M)
print(raw.label)
display(inv.style.format({"injected": "{:,.0f}"}).hide(axis="index"))

knrates_results_latest.zip (created 20261111T024800Z, git d7af6653)


suite,model,realisations,injected,used in main analysis
01 fixed-parameter ensemble,Bu2026Fixed,100,"100,000,000",True
01 fixed-parameter ensemble,Bu2026Vary,100,"100,000,000",True
01 fixed-parameter ensemble,Metzger,100,"100,000,000",False
07 GW170817 parameter set,GW170817_Bu2026_FixedView,100,"100,000,000",False
07 GW170817 parameter set,GW170817_Bu2026_Isotropic,100,"100,000,000",False
07 GW170817 parameter set,GW170817_Metzger_Blue,100,"100,000,000",False
07 GW170817 parameter set,GW170817_Metzger_Mean,100,"100,000,000",False
03 regimes x populations,AT2017gfo_FixedView,9,"9,000,000",False
03 regimes x populations,AT2017gfo_IsotropicView,9,"9,000,000",False
03 regimes x populations,BroadPopulation_Isotropic,9,"9,000,000",False


## What the results say

One row per claim. Values are recomputed on every run, so a new snapshot updates this table.

In [3]:
cmp = A.andreoni_comparison(ctx.tm, runs, M).set_index("model")
inc = A.inclination_efficiency(ctx.src, runs, ISO, n_bins=20)
xc = A.inclination_crosscheck(inc, lim, ISO, FIX)
dq = A.distance_quantiles(ctx.zh, FIX)
s_iso = ctx.src[ctx.src.model == ISO]
r07 = NB.all_limits(SNAPSHOT)
fid02 = ctx.snap["suite02_fiducial"].set_index("model")
resp = A.criteria_response(ctx.snap["suite02_slices_1d"], fid02)
top = resp[resp.model == FIX].sort_values("vt_max_over_min").iloc[-1]
s3 = ctx.snap["suite03_runs"].rename(columns={"population_type": "model"})
l3 = summarize_runs(s3, by=("criteria_regime", "model")).set_index(["criteria_regime", "model"])
a21 = L.lookup("andreoni2021_ztfrest_combined")
e = A.vt_by_epoch(ctx.tm, runs, FIX); e = e[e.vt_cumulative > 0]
upto = lambda d: e[e.month_end < pd.Timestamp(d) + pd.Timedelta(days=1)].iloc[-1].vt_cumulative   # through day d
a21_pred = L.lookup("andreoni2020_gw170817")["limit"] * upto(L.lookup("andreoni2020_gw170817")["data_end"]) / upto(a21["data_end"])

claims = [
    ("R95, AT2017gfo-like ejecta at iota = 25.8 deg [Gpc^-3 yr^-1]", f"{lim.loc[FIX, 'r95']:.0f}", "01 · §4, Table 1"),
    ("R95, same ejecta, isotropic viewing [Gpc^-3 yr^-1]", f"{lim.loc[ISO, 'r95']:.0f}", "01 · §4, Table 1"),
    ("⟨VT⟩ fixed / ⟨VT⟩ isotropic", f"{lim.loc[FIX, 'vt_gpc3_yr'] / lim.loc[ISO, 'vt_gpc3_yr']:.2f}", "01, 03 · §5"),
    ("Fixed-angle limit inside the GWTC-4 BNS rate range (7.6-250)?", str(7.6 <= lim.loc[FIX, 'r95'] <= 250), "01 · Fig. 3"),
    ("Isotropic limit inside the GWTC-4 BNS rate range?", str(7.6 <= lim.loc[ISO, 'r95'] <= 250), "01 · Fig. 3"),
    ("Half / 99% of recoveries within d_L [Mpc] (fixed angle)", f"{dq['dL50']:.0f} / {dq['dL99']:.0f}", "01 · Fig. 4"),
    ("MC uncertainty of pooled ⟨VT⟩ (fixed / isotropic)", f"{100 * lim.loc[FIX, 'vt_err_empirical'] / lim.loc[FIX, 'vt_gpc3_yr']:.2f}% / {100 * lim.loc[ISO, 'vt_err_empirical'] / lim.loc[ISO, 'vt_gpc3_yr']:.2f}%", "04 · §6"),
    ("R95 in Andreoni+2020's 23 months (fixed / iso)", f"{cmp.loc[FIX, 'r95_window']:.0f} / {cmp.loc[ISO, 'r95_window']:.0f}", "02 · §4.1"),
    ("... change vs. published 1775 / 4029", f"{100 * cmp.loc[FIX, 'window_change_vs_published']:+.0f}% / {100 * cmp.loc[ISO, 'window_change_vs_published']:+.0f}%", "02 · Fig. 1"),
    ("⟨VT⟩ growth, full survey vs. 23 months (duration alone)", f"{cmp.loc[FIX, 'vt_gain_full_over_window']:.2f}x ({cmp.loc[FIX, 'duration_gain']:.2f}x)", "02 · §4.1"),
    ("Andreoni+2020 limits carried to 8 yr (fixed / iso)", f"{cmp.loc[FIX, 'published_projected_full']:.0f} / {cmp.loc[ISO, 'published_projected_full']:.0f}", "02 · §4.1"),
    ("ZTFReST 900 vs. A20 carried to its data end", f"900 vs. {a21_pred:.0f}", "02"),
    ("Isotropic: share of injections / recoveries at iota > 45 deg", f"{100 * np.cos(np.pi / 4):.0f}% / {100 * (s_iso.inclination_rad > np.pi / 4).mean():.0f}%", "03 · §5"),
    ("⟨VT⟩(iota): first bin / last bin", f"{inc.sort_values('iota_deg_lo').vt.iloc[0] / inc.sort_values('iota_deg_lo').vt.iloc[-1]:.0f}x", "03 · Fig. 2"),
    ("Isotropic ⟨VT⟩(iota) at 25.8 deg vs. fixed-angle ensemble", f"{100 * xc['rel_diff']:+.2f}%", "03 · §5"),
]
if "GW170817_Bu2026_FixedView" in r07.index:
    claims.append(("Suite 07 (M_dyn −0.1 dex) ⟨VT⟩ change, fixed / iso",
                   f"{100 * (r07.loc['GW170817_Bu2026_FixedView', 'vt_gpc3_yr'] / lim.loc[FIX, 'vt_gpc3_yr'] - 1):+.1f}% / "
                   f"{100 * (r07.loc['GW170817_Bu2026_Isotropic', 'vt_gpc3_yr'] / lim.loc[ISO, 'vt_gpc3_yr'] - 1):+.1f}%", "04 · §3"))
claims += [
    ("Suite 02 filter / Suite 01 at the same criteria (fixed angle)", f"{fid02.loc[FIX, 'effective_vt_gpc3_yr'] / lim.loc[FIX, 'vt_gpc3_yr']:.2f}", "05"),
    ("Criterion with the largest ⟨VT⟩ range (fixed angle)", f"{top.criterion}: {top.vt_max_over_min:.1f}x", "05 · Fig. 6"),
]
if ("Baseline_ZTFReST", "BroadPopulation_Isotropic") in l3.index:
    claims.append(("Suite 03 broad population / AT2017gfo isotropic, ⟨VT⟩ (baseline criteria)",
                   f"{l3.loc[('Baseline_ZTFReST', 'BroadPopulation_Isotropic'), 'vt_gpc3_yr'] / l3.loc[('Baseline_ZTFReST', 'AT2017gfo_IsotropicView'), 'vt_gpc3_yr']:.2f}", "04 · §6"))
pd.set_option("display.max_colwidth", 90)
display(pd.DataFrame(claims, columns=["claim", "value", "notebook · paper"]).style.hide(axis="index"))

claim,value,notebook · paper
"R95, AT2017gfo-like ejecta at iota = 25.8 deg [Gpc^-3 yr^-1]",192,"01 · §4, Table 1"
"R95, same ejecta, isotropic viewing [Gpc^-3 yr^-1]",664,"01 · §4, Table 1"
⟨VT⟩ fixed / ⟨VT⟩ isotropic,3.46,"01, 03 · §5"
Fixed-angle limit inside the GWTC-4 BNS rate range (7.6-250)?,True,01 · Fig. 3
Isotropic limit inside the GWTC-4 BNS rate range?,False,01 · Fig. 3
Half / 99% of recoveries within d_L [Mpc] (fixed angle),143 / 275,01 · Fig. 4
MC uncertainty of pooled ⟨VT⟩ (fixed / isotropic),0.23% / 0.45%,04 · §6
R95 in Andreoni+2020's 23 months (fixed / iso),1069 / 3684,02 · §4.1
... change vs. published 1775 / 4029,-40% / -9%,02 · Fig. 1
"⟨VT⟩ growth, full survey vs. 23 months (duration alone)",5.57x (4.18x),02 · §4.1


## Open items in the manuscripts

Every red `\tbd{...}` placeholder, read straight from the LaTeX sources.

In [4]:
import re
rows = []
for tex in ["paper/ms.tex", "paper/companion_paper_imposters_criteria.tex"]:
    text = (ROOT / tex).read_text()
    for mt in re.finditer(r"\\tbd\{", text):
        depth, i = 1, mt.end()
        while depth and i < len(text):
            depth += {"{": 1, "}": -1}.get(text[i], 0); i += 1
        line = text.count("\n", 0, mt.start()) + 1
        if text[text.rfind("\n", 0, mt.start()) + 1:mt.start()].lstrip().startswith("%"):
            continue   # LaTeX comment
        body = " ".join(text[mt.end():i - 1].split())
        if body != "#1":
            rows.append(dict(file=tex.split("/")[-1], line=line, placeholder=body[:160] + ("…" if len(body) > 160 else "")))
display(pd.DataFrame(rows).style.hide(axis="index"))

file,line,placeholder
ms.tex,36,co-authors
ms.tex,39,"Population-averaged limit (Suite 03 re-run), real-data candidate search result, and criteria-systematics band (Suite 02 re-run)."
ms.tex,52,Section on the archival candidate search (Section~\ref{sec:search}).
ms.tex,60,companion citation
ms.tex,67,citation/software DOI
ms.tex,75,$N_{\rm obs}$ from the archival search; the limits below assume $N_{\rm obs}=0$.
ms.tex,79,Replace the ad hoc parameter set by a published AT2017gfo posterior \citep[e.g.][]{Anand2023} and marginalise over it.
ms.tex,80,"To be removed: compared with AT2017gfo photometry, the ``blue'' and ``mean'' presets are 1--3\,mag too bright and the $\kappa=398$ template is $\approx0.7$\,mag…"
ms.tex,80,Re-tune or drop the one-zone models.
ms.tex,81,Suite~03 re-run: per-transient parameter draws (\texttt{Bu2026KilonovaPopulation}) and isotropic $\iota$; prior choice to be justified against numerical-relativ…
